# Практика к занятию "Рекурентные сети 2"

**Задача:** Генерация текста
1. Скачать датасет
1. Привести в нижний регистр, оставить только английские буктвы и пробелы
1. Подготовить датасет
    - X - это 40 букв,
    - Y - это 41-я буква,
    - Шаг = 3
1. Обучить модель предстказывать Y
    Параметры модели: embeding (64), GRU (128), output
1. На каждой эпохе генерировать часть текста: ена выходе - случайные 40 последовательных символов текста, следующие 40 символов генерирует нейросеть

In [1]:
import torch
from torch import nn
import re
import random
import tqdm
import time

## Данные

Скачиваем и предобрабатываем данные (датасет текстов Ницше)
1. Предобработать данные
1. Разбить на токены
1. Закодировать токены
1. Упаковать батчи

In [3]:
!wget https://s3.amazonaws.com/text-datasets/nietzsche.txt



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://s3.amazonaws.com/text-]87Saving 'nietzsche.txt'
87nietzsche.txt         63% [==================>           ]  373.62K    --.-KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87nietzsche.txt        100% [=============================>]  586.81K    9.91MB/s87HTTP response 200 OK [https://s3.amazonaws.com/text-datasets/nietzsche.txt]
87nietzsche.txt        100% [=============================>]  586.81K    9.91MB/s87[Files: 1  Bytes: 586.81K [574.]8

### Первичная предобработка

In [4]:
with open('nietzsche.txt', encoding='utf-8') as f:
    text = f.read().lower()
print('length:', len(text))    
text = re.sub('[^a-z]', ' ', text)
text = re.sub('\s+', ' ', text)

length: 600893


In [5]:
text[:100]

'preface supposing that truth is a woman what then is there not ground for suspecting that all philos'

### Создаем словарь из токенов (символов)

In [6]:
INDEX_TO_CHAR = sorted(list(set(text)))
CHAR_TO_INDEX = {c: i for i, c in enumerate(INDEX_TO_CHAR)}

In [7]:
CHAR_TO_INDEX

{' ': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

### Разбиваем текст на кусочки по 40 символов
Один такой кусочек назовем предложением, когда символ с индексом 41 будет целевой переменной - следующим токеном в предложении

Будем генерировать предложения из датасета со сдвигом в 3 символа

In [18]:
MAX_LEN = 40
STEP = 3
SENTENCES = []
NEXT_CHARS = []

for i in range(0, len(text) - MAX_LEN, STEP):
    SENTENCES.append(text[i: i + MAX_LEN])
    NEXT_CHARS.append(text[i + MAX_LEN])
print('Num sents:', len(SENTENCES))   

Num sents: 193075


In [20]:
for i in range(10):
    print(SENTENCES[i], '|', NEXT_CHARS[i])

preface supposing that truth is a woman  | w
face supposing that truth is a woman wha | t
e supposing that truth is a woman what t | h
upposing that truth is a woman what then |  
osing that truth is a woman what then is |  
ng that truth is a woman what then is th | e
that truth is a woman what then is there |  
t truth is a woman what then is there no | t
ruth is a woman what then is there not g | r
h is a woman what then is there not grou | n


### Закодируем токены числами и положим их в pythorch векторы

In [21]:
print('Vectorization...')
X = torch.zeros((len(SENTENCES), MAX_LEN), dtype=int) # размер ваборки * длина предложения
Y = torch.zeros((len(SENTENCES)),dtype=int) # размер выборки

for i, sentence in enumerate(SENTENCES):
    for t, char in enumerate(sentence):
        X[i, t] = CHAR_TO_INDEX[char]
    Y[i] = CHAR_TO_INDEX[NEXT_CHARS[i]]    

Vectorization...


In [35]:
for n, i in enumerate(range(5), 1):
    print(f"{n} : X = {X[i]}, Y = {Y[i]}")

1 : X = tensor([16, 18,  5,  6,  1,  3,  5,  0, 19, 21, 16, 16, 15, 19,  9, 14,  7,  0,
        20,  8,  1, 20,  0, 20, 18, 21, 20,  8,  0,  9, 19,  0,  1,  0, 23, 15,
        13,  1, 14,  0]), Y = 23
2 : X = tensor([ 6,  1,  3,  5,  0, 19, 21, 16, 16, 15, 19,  9, 14,  7,  0, 20,  8,  1,
        20,  0, 20, 18, 21, 20,  8,  0,  9, 19,  0,  1,  0, 23, 15, 13,  1, 14,
         0, 23,  8,  1]), Y = 20
3 : X = tensor([ 5,  0, 19, 21, 16, 16, 15, 19,  9, 14,  7,  0, 20,  8,  1, 20,  0, 20,
        18, 21, 20,  8,  0,  9, 19,  0,  1,  0, 23, 15, 13,  1, 14,  0, 23,  8,
         1, 20,  0, 20]), Y = 8
4 : X = tensor([21, 16, 16, 15, 19,  9, 14,  7,  0, 20,  8,  1, 20,  0, 20, 18, 21, 20,
         8,  0,  9, 19,  0,  1,  0, 23, 15, 13,  1, 14,  0, 23,  8,  1, 20,  0,
        20,  8,  5, 14]), Y = 0
5 : X = tensor([15, 19,  9, 14,  7,  0, 20,  8,  1, 20,  0, 20, 18, 21, 20,  8,  0,  9,
        19,  0,  1,  0, 23, 15, 13,  1, 14,  0, 23,  8,  1, 20,  0, 20,  8,  5,
        14,  0,  9, 19]), Y = 

### Упакуем предложения в батчи

In [36]:
BATCH_SIZE = 512
dataset = torch.utils.data.TensorDataset(X, Y)
data = torch.utils.data.DataLoader(
    dataset=dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

## LSTM & GRU

In [40]:
X.shape

torch.Size([193075, 40])

In [41]:
embedding = nn.Embedding(len(INDEX_TO_CHAR), 15)
rnn = nn.LSTM(15, 128, batch_first=True)

Применим оба слоя к кусочку датасета

In [42]:
o, s = rnn(embedding(X[0:10]))
o.shape, len(s), s[0].shape, s[1].shape

(torch.Size([10, 40, 128]),
 2,
 torch.Size([1, 10, 128]),
 torch.Size([1, 10, 128]))

In [43]:
rnn = nn.GRU(15, 128, batch_first=True)
o, s = rnn(embedding(X[0:10]))
o.shape, len(s), s[0].shape

(torch.Size([10, 40, 128]), 1, torch.Size([10, 128]))

### Архитектура нейросети

Строим нейросеть с одним рекуррентным слоем, которая будет предсказывать следующий токен

In [58]:
class NeuralNetwork(nn.Module):
    def __init__(self, rnnClass, dictionary_size, embedding_size, num_hiddens, num_classes):
        super().__init__()

        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(dictionary_size, embedding_size)
        self.hidden = rnnClass(embedding_size, num_hiddens, batch_first=True)
        self.output = nn.Linear(num_hiddens, num_classes)

    def forward(self, X):
        out = self.embedding(X)
        _, state = self.hidden(out)
        prediction = self.output(state[0].squeeze())
        return prediction

Инициализируем модель с GRU слоем

In [59]:
model = NeuralNetwork(nn.GRU, len(CHAR_TO_INDEX), 64, 128, len(CHAR_TO_INDEX))

Инициализируем модель с LSTM слоем

In [70]:
model = NeuralNetwork(nn.LSTM, len(CHAR_TO_INDEX), 64, 128, len(CHAR_TO_INDEX))

In [71]:
X.shape

torch.Size([193075, 40])

In [72]:
model(X[0:1])

tensor([ 0.0409, -0.0163, -0.0735, -0.0886, -0.0024, -0.0208,  0.0062,  0.1002,
         0.0783,  0.0928,  0.0677,  0.0510,  0.0830,  0.0004, -0.0277,  0.0951,
        -0.0869,  0.0110, -0.0052,  0.0112, -0.0916, -0.0013,  0.0805,  0.0418,
         0.0110, -0.0316, -0.0674], grad_fn=<ViewBackward0>)

In [73]:
model = model.cuda()

Функция для семплирования токенов

In [74]:
def sample(preds):
    softmaxed = torch.softmax(preds, 0)
    probas = torch.distributions.multinomial.Multinomial(1, softmaxed).sample()
    return probas.argmax()

Функция генерации текста

In [75]:
def generate_text():
    # Берем случайный отрезок текста нужной длины
    start_index = random.randint(0, len(text) - MAX_LEN - 1)

    generated = ''
    sentence = text[start_index: start_index + MAX_LEN]
    generated += sentence

    # Кодируем токены числами
    for i in range(MAX_LEN):
        x_pred = torch.zeros((1, MAX_LEN), dtype=int)
        for t, char in enumerate(generated[-MAX_LEN:]):
            x_pred[0, t] = CHAR_TO_INDEX[char]
        # предсказываем следующий символ
        preds = model(x_pred.cuda()).cpu()
        next_char = INDEX_TO_CHAR[sample(preds)]    
        generated = generated + next_char

    print(generated[:MAX_LEN] + '|' + generated[MAX_LEN:])    


In [76]:
generate_text()

er largely into our self satisfaction on|vzoyafbjhgjplwib qw gsyc mvqzanyqeind kv


### Обучение модели

In [77]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [78]:
for ep in range(30):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    model.train()
    for X_b, y_b in data:
        X_b, y_b = X_b.cuda(), y_b.cuda()
        optimizer.zero_grad()
        answer = model(X_b)
        loss = criterion(answer, y_b)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    print("Epoch {}. Time: {:.3f}, Train loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))    
    model.eval()
    generate_text()

Epoch 0. Time: 2.502, Train loss: 2.174
l valuations in forerunners in men of th|ousplit the itumaes of the care d diedta
Epoch 1. Time: 2.474, Train loss: 1.793
r multifarious and boisterous art whithe|d and thas pocfulualy in obzencal ingera
Epoch 2. Time: 2.404, Train loss: 1.655
something perfectly obnoxious or ludicro|r to her the morthing in now peses arist
Epoch 3. Time: 2.455, Train loss: 1.569
wit the innate methodology and relations| muterings wich the the futh of be fiarn
Epoch 4. Time: 2.475, Train loss: 1.509
 effort or constraint a downwards withou|ld he sen do a restifit be noscend us at
Epoch 5. Time: 2.411, Train loss: 1.463
 backward spirits appear who try to conj|uregen tranks there is grade at makany n
Epoch 6. Time: 2.474, Train loss: 1.427
 company of beings under whose hands gla|te a rope in the peciside but is to has 
Epoch 7. Time: 2.465, Train loss: 1.395
 treated with love even with infatuation| for the lolg of extrustral expliciate p
Epoch 8. Time: 2.409, Tr